In [ ]:

import kagglehub
import pandas as pd
import numpy as np
import os

# Load dataset
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
csv_path = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(csv_path)

# Preserve original dataset
raw_data = df.copy()
df = raw_data.copy()

# 1. Remove duplicates
print("\n========== DUPLICATES ==========")
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()

# 2. Fix data types
print("\n========== DATA TYPES ==========")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

print(df.loc[
    df["TotalCharges"].isna(),
    ["tenure", "MonthlyCharges", "Churn"]
])

# 3. Handle missing TotalCharges
# These records have tenure = 0, so no charges have accumulated.
df.loc[df["TotalCharges"].isna(), "TotalCharges"] = 0

print("Missing TotalCharges after cleaning:",
      df["TotalCharges"].isna().sum())

# 4. Standardise SeniorCitizen
df["SeniorCitizen"] = df["SeniorCitizen"].map({
    0: "No",
    1: "Yes"
})

# 5. Check results
print("\n========== CLEANED DATA ==========")
print(df.info())
print("\nRemaining duplicates:", df.duplicated().sum())
print("\nRemaining missing values:")
print(df.isna().sum()[df.isna().sum() > 0])

# 6. Exploratory data analysis
print("\n========== NUMERICAL SUMMARY ==========")
print(df.describe())

# 7. Churn patterns
print("\n========== CHURN BY CATEGORY ==========")

categories = [
    "Contract", "InternetService", "PaymentMethod",
    "TechSupport", "OnlineSecurity", "SeniorCitizen",
    "PaperlessBilling", "Partner"
]

for col in categories:
    churn_rates = (
        df.groupby(i, observed=True)["Churn"]
        .apply(lambda x: x.eq("Yes").mean() * 100)
        .sort_values(ascending=False)
        .round(1)
    )

    print(f"\n{i}:")
    print(churn_rates.to_string())


print('#======================feature engineering============#')
services = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
# feature 1: number of services subscribed (InternetService != 'No' counts as one)
df['num_services'] = (df[services]=='Yes').sum(axis=1) + (df['InternetService']!='No').astype(int)

# Feature 2: average charge per month of tenure (if tenure ==0 ; use MonthlyCharges)
df['avg_monthly_spend'] = np.where(df['tenure']>0, df['TotalCharges']/df['tenure'].replace(0,np.nan), df['MonthlyCharges'])

print(df[['num_services','avg_monthly_spend']].head())




print('============Patterns Observed=============')
print('''
1. Month-to-Month customers churn more than one/two year customers.
2. Newer customers tend to leave most 
3. fibre optics customers churn more 
4. EFT payers churn more
5. senior citizens churn more than non seniors

''')

#save the cleaned csv
raw_data.to_csv('cleaned_data.csv', index=False)
print('saved', df.shape)



========== DUPLICATES ==========
Duplicates: 0

========== DATA TYPES ==========
Missing TotalCharges: 11
      tenure  MonthlyCharges Churn
488        0           52.55    No
753        0           20.25    No
936        0           80.85    No
1082       0           25.75    No
1340       0           56.05    No
3331       0           19.85    No
3826       0           25.35    No
4380       0           20.00    No
5218       0           19.70    No
6670       0           73.35    No
6754       0           61.90    No
Missing TotalCharges after cleaning: 0

========== CLEANED DATA ==========
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   str    
 3   Partner           7043 non-null   str    
 4   Dependents     